# Import UN Trade and Development (UNCTAD) API data

- Author: Bryan Bravo
- Created: 2026-03-24
## Import Libraries

In [1]:
import os
import sys
import gzip
import io

os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")
# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)


# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

### Variables

In [3]:
end_date = proj_vars.end_date
CLIENT_ID = hardcoded_keys.UNCTAD_CLIENT_ID
CLIENT_SECRET = hardcoded_keys.UNCTAD_API_KEY
TEMP_FILE_PATH = "import_datasets/synchrone.csv.gz"  # currently saving to local environment


## Query

In [4]:
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    # 'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}


### Import LSCI 
https://unctadstat.unctad.org/datacentre/reportInfo/US.LSCI_M

In [8]:
# Configuration
URL = "https://unctadstat-user-api.unctad.org/US.LSCI_M/cur/Facts?culture=en"

# The same filter string from your R script (kept as a single string)
FILTER = (
    "Economy/Code in ('008','012','016','024','660','028','032','533','036','044','048','050',"
    "'052','056','084','204','060','535','076','092','096','100','132','116','120','124','136',"
    "'152','156','344','158','162','166','170','174','178','184','188','384','191','192','531',"
    "'196','408','180','208','262','212','214','218','818','222','226','232','233','238','234',"
    "'242','246','250','254','258','266','270','268','276','288','292','300','304','308','312',"
    "'316','320','831','324','624','328','332','340','352','356','360','364','368','372','376',"
    "'380','388','392','832','400','404','296','414','428','422','430','434','440','450','458',"
    "'462','470','584','474','478','480','175','484','583','499','500','504','508','104','516',"
    "'520','528','530','540','554','558','566','570','574','580','578','512','586','585','591',"
    "'598','600','604','608','616','620','630','634','410','498','638','642','643','654','659',"
    "'662','666','670','882','678','682','686','891','690','694','702','534','705','090','706',"
    "'710','724','144','729','736','740','752','760','764','626','768','776','780','788','792',"
    "'796','798','804','784','826','834','840','850','858','548','862','704','876','887') and "
    "Month/Code in ('2006M02','2006M05','2006M08','2006M11','2007M02','2007M05','2007M08','2007M11',"
    "'2008M02','2008M05','2008M08','2008M11','2009M02','2009M05','2009M08','2009M11','2010M02',"
    "'2010M05','2010M08','2010M11','2011M02','2011M05','2011M08','2011M11','2012M02','2012M05',"
    "'2012M08','2012M11','2013M02','2013M05','2013M08','2013M11','2014M02','2014M05','2014M08',"
    "'2014M11','2015M02','2015M05','2015M08','2015M11','2016M02','2016M05','2016M08','2016M11',"
    "'2017M02','2017M05','2017M08','2017M11','2018M02','2018M05','2018M08','2018M11','2019M02',"
    "'2019M05','2019M08','2019M11','2020M02','2020M05','2020M08','2020M11','2021M02','2021M05',"
    "'2021M08','2021M11','2022M02','2022M05','2022M08','2022M11','2023M01','2023M02','2023M03',"
    "'2023M04','2023M05','2023M06','2023M07','2023M08','2023M09','2023M10','2023M11','2023M12',"
    "'2024M01','2024M02','2024M03','2024M04','2024M05','2024M06','2024M07','2024M08','2024M09',"
    "'2024M10','2024M11','2024M12','2025M01','2025M02','2025M03','2025M04','2025M05','2025M06',"
    "'2025M07','2025M08','2025M09','2025M10','2025M11','2025M12','2026M01','2026M02','2026M03')"
)

FORM = {
    "$select": "Economy/Label ,Month/Label, Index_Average_M2_2023__100_Value, Index_Average_M2_2023__100_Footnote, Index_Average_M2_2023__100_MissingValue",
    "$filter": FILTER,
    "$orderby": "Economy/Order asc ,Month/Order asc",
    "$compute": "round(M6048/Value div 1, 2) as Index_Average_M2_2023__100_Value, M6048/Footnote/Text as Index_Average_M2_2023__100_Footnote, M6048/MissingValue/Label as Index_Average_M2_2023__100_MissingValue",
    "$format": "csv",
    "compress": "gz",
}

HEADERS = {
    "ClientId": CLIENT_ID,
    "ClientSecret": CLIENT_SECRET,
    "User-Agent": "python-unctad-client/1.0",
}

# Fetch data and load directly into memory
try:
    print("Requesting UNCTAD LSCI data...")
    response = requests.post(URL, data=FORM, headers=HEADERS, stream=True, timeout=60)
    response.raise_for_status()
    
    # Decompress gzipped response in-memory
    decompressed_data = gzip.decompress(b"".join(response.iter_content(chunk_size=8192)))
    
    # Read CSV directly from decompressed bytes without writing to disk
    lsci_df = pd.read_csv(io.BytesIO(decompressed_data), header=0, na_values="", encoding="utf-8")
    
    # Standardize column names
    lsci_df.columns = [c.strip().lower() for c in lsci_df.columns]
    
    print("Loaded DataFrame with shape:", lsci_df.shape)
    print(lsci_df.head().to_string(index=False))
    
    # Select and rename relevant columns
    lsci_df = lsci_df[['economy_label', 'month_label', 'index_average_m2_2023__100_value']]
    lsci_df.columns = ['country', 'month_label', 'lsci']
    
except requests.HTTPError as e:
    print("HTTP error:", e, file=sys.stderr)
except Exception as e:
    print("Error:", e, file=sys.stderr)


Requesting UNCTAD LSCI data...
Loaded DataFrame with shape: (19364, 5)
economy_label month_label  index_average_m2_2023__100_value  index_average_m2_2023__100_footnote index_average_m2_2023__100_missingvalue
      Albania   Nov. 2006                              5.01                                  NaN                                     NaN
      Albania   Feb. 2007                              5.48                                  NaN                                     NaN
      Albania   May  2007                              5.48                                  NaN                                     NaN
      Albania   Aug. 2007                              5.04                                  NaN                                     NaN
      Albania   Nov. 2007                              5.19                                  NaN                                     NaN


In [9]:
# lowercase country and w/trim country and month_label
lsci_df['country'] = lsci_df['country'].str.lower().str.strip()
lsci_df['month_label'] = lsci_df['month_label'].str.lower().str.strip()

# Extract year and month from month_label
lsci_df[['month', 'year']] = lsci_df['month_label'].str.split('. ', expand=True)
lsci_df['year'] = lsci_df['year'].astype(int)
lsci_df['month'] = lsci_df['month'].map({
    mon: i+1 
    for i, mon in enumerate(['jan', 'feb', 'mar', 'apr', 'ma', 'jun',
                             'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])
}).astype(int)

# Filter for countries in the analysis
lsci_df = lsci_df[lsci_df['country'].isin(country_name for country_name in country_mapping.keys())]

# Convert to Spark Df
lsci_df = spark.createDataFrame(lsci_df[['country', 'year', 'month', 'lsci']])
lsci_df.repartition(10).cache().count()

1177

In [10]:
lsci_df.dtypes

[('country', 'string'),
 ('year', 'bigint'),
 ('month', 'bigint'),
 ('lsci', 'double')]